# pandas

## 基本

In [44]:
# テスト用データ生成
import pandas as pd
import copy

data = [['A', 100], ['B', 30], ['C', 150]]
_df = pd.DataFrame(data, columns=['name', 'value'])
_df

,name,value
0,A,100
1,B,30
2,C,150


TODO
* column操作系
* index操作系
* concat系
* merge系
* loc, iloc 値の取り出し　値の上書き
* 列単位への関数適応



In [45]:
# 特定の列をインデックスに指定する
df = copy.deepcopy(_df)
df.set_index('name')

,value
name,
A,100
B,30
C,150


In [46]:
# columnsの名前を変更する
df = copy.deepcopy(_df)
df.rename(columns={'name':'city','value':'number'})

,city,number
0,A,100
1,B,30
2,C,150


In [47]:
# カラム,インデックスをリストで取得する
df = copy.deepcopy(_df)
col = list(df.columns)
ind = list(df.index)
print(f'{col=}')
print(f'{ind=}')


col=['name', 'value']
ind=[0, 1, 2]


行、列を追加する場合、以下の3手段をとることができる。
* 値を一括で追加（単一）
* 値をリストで指定して追加（リスト）
* 不足分はNanで代入する（series）

concatも使用できるが、今回は除外する。
<https://takilog.com/pandas-dataframe-append-concat/>

In [48]:
# 列の追加 値は一括で入力
df = copy.deepcopy(_df)
df['test'] = 0
df

,name,value,test
0,A,100,0
1,B,30,0
2,C,150,0


In [49]:
# 列の追加 値はlistで入力
df = copy.deepcopy(_df)
df['test'] = [1, 2, 3]
df

,name,value,test
0,A,100,1
1,B,30,2
2,C,150,3


In [50]:
# 列の追加 値が足りない場合はNanで追加
df = copy.deepcopy(_df)
data = [1, 3]
sr = pd.Series(data, index=[0, 2])
df['test'] = sr
df

,name,value,test
0,A,100,1.0
1,B,30,NaN
2,C,150,3.0


In [51]:
# 行を追加する　値は一括
df = copy.deepcopy(_df)
df.loc['s'] = 0
# df.loc['s', :] = 0
df

,name,value
0,A,100
1,B,30
2,C,150
s,0,0


In [52]:
# 行を追加する　値はlistで指定
df = copy.deepcopy(_df)
df.loc['s'] = ['d', 34]
# df.loc['s', :] = ['d', 34]
df

,name,value
0,A,100
1,B,30
2,C,150
s,d,34


In [53]:
# 行を追加する　series
df = copy.deepcopy(_df)
data = ['f', 32]
sr = pd.Series(data, index=['name', 'value'])

data = ['f']
sr = pd.Series(data, index=['name'])

df.loc[3] = sr
df

,name,value
0,A,100.0
1,B,30.0
2,C,150.0
3,f,NaN


## [tips] iterrowsと処理速度

<https://qiita.com/141sksk/items/9883be05a3851c90d1d1>

テスト用データ準備

In [54]:
from sklearn.datasets import load_iris
iris = load_iris()

In [55]:
iris.data.shape

(150, 4)

In [56]:
import pandas as pd
df_iris = pd.DataFrame(iris.data, columns=iris.feature_names)

In [57]:
df_iris.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


iterrowsは便利で分かりやすいが、遅い、、

In [58]:
%%timeit
for idx, row in df_iris.iterrows():
    if row[0] < 4:
        row[0] = 1


1.59 ms ± 44.6 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


listに変換すると、かなり速い

In [59]:
list_iris = df_iris.values.tolist()
list_iris[0:3]

[[5.1, 3.5, 1.4, 0.2], [4.9, 3.0, 1.4, 0.2], [4.7, 3.2, 1.3, 0.2]]

In [60]:
%%timeit
for ele in list_iris:
    if ele[0] < 4:
        ele[0] = 1

3.02 µs ± 38.3 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


dictでもだいぶ速い

In [61]:
dict_iris = df_iris.to_dict()
len(dict_iris.values())
dict_iris.keys()


dict_keys(['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)'])

In [62]:
%%timeit
for key, val in df_iris.to_dict().items():
    if key == 'sepal length (cm)':
        val[0] = 3

85.7 µs ± 1.2 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


applyもdictくらい速い

In [63]:
def map_oz(col):
    if col < 4:
        return 1
    return 0

In [64]:
%%timeit
df_iris['petal length (cm)'].apply(map_oz)

64.8 µs ± 1.73 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


---

case: 二つの表があり、片方の表に、もう片方の情報を反映させたいとき

In [65]:
# test
data1 = {'name': ['A', 'B', 'C', 'D'],
        'age': [10, 20, 30, 40],
        'skill': [0, 0, 0, 0],
        'flg': [0, 0, 0, 0]}

df1 = pd.DataFrame(data1)
df1


,name,age,skill,flg
0,A,10,0,0
1,B,20,0,0
2,C,30,0,0
3,D,40,0,0


In [66]:
data2 = {'name': ['A', 'C'],
        'skill': [22, 33]}

df2 = pd.DataFrame(data2)
df2


,name,skill
0,A,22
1,C,33


In [67]:
# patern1
# data1のname列をindexにして、data2をindexに合わせて更新
df1.set_index('name', inplace=True)
df2.set_index('name', inplace=True)

# data2の値でdata1の'skill'を更新し、該当する行の'flg'を1にする
df1.update(df2)
df1.loc[df2.index, 'flg'] = 1

# インデックスをリセットして元の形式に戻す
df1.reset_index(inplace=True)

df1

,name,age,skill,flg
0,A,10,22,1
1,B,20,0,0
2,C,30,33,1
3,D,40,0,0


---

groupby 練習

- メモ
  - データのkeyとなるカラムを複数作成し、groupbyで検索するときに使う
  - aggを活用することで統計データなどを一括で確認する

<https://note.nkmk.me/python-pandas-agg-aggregate/>  
<https://zenn.dev/yuto_mo/articles/fb1d010b30afff>

In [68]:
# テストデータ
data = {'Category': ['A', 'B', 'A', 'B', 'C', 'A', 'B', 'C'],
        'Values': [10, 20, 15, 25, 30, 5, 40, 50],
        }
df = pd.DataFrame(data)
df.head(3)

,Category,Values
0,A,10
1,B,20
2,A,15


In [69]:
# 合計
df.groupby('Category').sum()
df.groupby('Category', as_index=False).sum()  # 行のインデックス付与

,Category,Values
0,A,30
1,B,85
2,C,80


In [70]:
# 複数の列指定も可能
data = {'Category': ['apple', 'banana', 'apple', 'banana', 'orange', 'apple', 'banana', 'orange'],
        'Place': ['Japan', 'Japan', 'Japan', 'Japan', 'US', 'US', 'US', 'US'],
        'Num': [10, 20, 15, 25, 30, 5, 40, 50],
        }
df = pd.DataFrame(data)

df.groupby(['Category', 'Place']).sum()

Num
Category Place     
apple    Japan   25
         US       5
banana   Japan   45
         US      40
orange   US      80

In [71]:
# aggを利用することで、関数をまとめて適応できる
# df.groupby(['Category', 'Place']).agg(['sum', 'mean', 'max'])
# df.groupby(['Category', 'Place']).agg({'Num': ['sum', 'mean', 'max']})  # 列を指定して集計する場合
df.groupby(['Category', 'Place']).agg({'Num': ['sum', 'mean', 'max'], 'Place': ['count']})  # 列を指定して集計する場合

Num           Place
               sum  mean max count
Category Place                    
apple    Japan  25  12.5  15     2
         US      5   5.0   5     1
banana   Japan  45  22.5  25     2
         US     40  40.0  40     1
orange   US     80  40.0  50     2

---

apply練習
- apply 行、列に対して、処理を実行できる
- applymap 各要素に対して、処理を実行できる


In [72]:
# サンプルデータの作成
data = {'A': [1, 2, 3], 'B': [4, 5, 6], 'C': [7, 8, 9]}
df = pd.DataFrame(data)
df

,A,B,C
0,1,4,7
1,2,5,8
2,3,6,9


In [73]:
# 各列の最大値
df.apply(max, axis=0)

A    3
B    6
C    9
dtype: int64

In [74]:
# 各行の最小値
df.apply(min, axis=1)

0    1
1    2
2    3
dtype: int64

In [75]:
# 指定値以上の値を1に置き換える
df.applymap(lambda x: 1 if x > 5 else 0)

,A,B,C
0,0,0,1
1,0,0,1
2,0,1,1


---

pandasの日付型について

- csvデータから読み込んだ日付データはObject型として読み込まれる
- 意図的にdatetime型に変換することで扱いを容易にする

<https://qiita.com/takubb/items/e18ea4f7c4ecc8be4a5f>

In [80]:
df = pd.read_csv('test_date.csv')
df

,date,values
0,2025-01-01,1
1,2025-01-02,2
2,2025-01-03,3


In [81]:
df['date'].values[0]

'2025-01-01'

In [82]:
df['date'] = pd.to_datetime(df['date'], format='%Y-%m-%d')
df['date'].values[0]

numpy.datetime64('2025-01-01T00:00:00.000000000')